#### для работы в google colab

In [3]:
!gdown 1bHtAfjpphYjjG45Y0iOsy7kQr4eplA2H

Downloading...
From: https://drive.google.com/uc?id=1bHtAfjpphYjjG45Y0iOsy7kQr4eplA2H
To: /kaggle/working/Regression_dataset.zip
100%|██████████████████████████████████████| 13.8M/13.8M [00:00<00:00, 39.4MB/s]


In [5]:
import zipfile

zip_path = '/kaggle/working/Regression_dataset.zip'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/kaggle/working/regression_dataset')

print("📁 Архив распакован.")


📁 Архив распакован.


In [6]:
import pandas as pd

In [29]:
# для google colab

images_folder = '/kaggle/working/regression_dataset/train/'
test_folder = '/kaggle/working/regression_dataset/test/'

df = pd.read_csv('/kaggle/working/regression_dataset/train.csv')
test = pd.read_csv('/kaggle/working/regression_dataset/test.csv')

df.head()

,likes,reposts,views,filename,url
0,3854.0,6578.0,1051097.0,935.jpg,https://sun9-46.userapi.com/s/v1/if2/TO7aTYbdN...
1,374.0,476.0,86354.0,290.jpg,https://sun9-85.userapi.com/s/v1/ig2/RGUtvk0yD...
2,135.0,67.0,54483.0,544.jpg,https://sun9-36.userapi.com/s/v1/ig2/9H6EEHaUA...
3,240.0,1228.0,191603.0,275.jpg,https://sun9-57.userapi.com/s/v1/ig2/_3Xhtrcnf...
4,225.0,220.0,105902.0,109.jpg,https://sun9-42.userapi.com/s/v1/ig2/mmvCLMYDp...


### Выгружаем данные и убираем лишние классы

In [30]:
import pandas as pd
import numpy as np

In [1]:
# для локального запуска
import os

if os.getcwd().split('/')[-1] != 'memobot':
    os.chdir('..')
os.getcwd()


'/Users/mhlgvr/Documents/Yandex.Disk.localized/CU2/AI/Bootcamps 2025/memobot'

In [44]:
# images_folder = 'data/regression_dataset/train/'
# test_folder = 'data/regression_dataset/test/'

# df = pd.read_csv('data/regression/train.csv')
# test = pd.read_csv('data/regression/test.csv')

# df.head()

## Собираем датасет

In [45]:
from torch.utils.data import Dataset, DataLoader
import torch
from PIL import Image

class VkDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['filename'])
        image = Image.open(img_path).convert('RGB')
        targets = torch.Tensor([row['likes'], row['reposts'], row['views']])

        if self.transform:
            image = self.transform(image)

        return image, targets


In [46]:
from torch.utils.data import random_split
from torchvision import transforms

torch.manual_seed(42)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

train_data = VkDataset(df, images_folder, transform=transform)
test_data = VkDataset(test, test_folder, transform=transform)

train_ds, val_ds = random_split(train_data, [0.8, 0.2])

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=32, shuffle=False)

test_dl = DataLoader(test_data, batch_size=32, shuffle=False)


In [47]:
len(train_data), len(test_data)

(759, 190)

### Быстрые функции обучения и тестирования модели

In [71]:
import torch.nn as nn
from tqdm import tqdm


def train_model(model, model_path, n_epochs=5, batch_size=32, optimizer=None, criterion=None):
    if not optimizer:
        optimizer = torch.optim.Adam(model.parameters())
    if not criterion:
        criterion = nn.MSELoss()

    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_dl = DataLoader(test_data, batch_size=batch_size, shuffle=False)
    
    best_loss = float('inf')
    
    
    for epoch in range(n_epochs):
        model.train()
        total_loss = 0
    
        for image, targets in tqdm(train_dl, desc=f"Epoch {epoch+1} [Train]", leave=False):
            image, targets = image.to(device), targets.to(device)
            preds = model(image)
            loss = criterion(preds, targets)
    
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
            total_loss += loss.item()
    
        model.eval()
        val_loss = 0
    
        for image, targets in tqdm(val_dl, desc=f"Epoch {epoch+1} [Val]", leave=False):
            image, targets = image.to(device), targets.to(device)
            with torch.no_grad():
                preds = model(image)
                loss = criterion(preds, targets)
            
            val_loss += loss.item()
    
        val_loss /= len(val_dl)
        total_loss /= len(train_dl)
    
        if val_loss < best_loss:
            best_loss = val_loss
            torch.save(model.state_dict(), model_path)
    
        print(f"Epoch {epoch+1} [Train] Loss: {total_loss:.4f}, [Val] Loss: {val_loss:.4f}")

In [72]:
def test_model(model, criterion):
    test_loss = 0
    
    for image, targets in tqdm(test_dl, desc=f"Epoch {epoch+1} [Test]"):
        image, targets = image.to(device), targets.to(device)
        with torch.no_grad():
            preds = model(image)
            loss = criterion(preds, targets)
        test_loss += loss.item()
    
    test_loss /= len(test_dl)
    
    print(f"Test Loss: {test_loss:.4f}")

In [ ]:
transform = transforms.Compose([
    transforms.Resize(64),
    transforms.CenterCrop(64),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])


In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3),
            nn.ReLU(),
            nn.Conv2d(64, 128, 3),
            nn.Flatten()
        )
        self.fc_mu = nn.Linear(128*8*8, latent_dim)
        self.fc_logvar = nn.Linear(128*8*8, latent_dim)

        self.decoder_input = nn.Linear(latent_dim, 128*8*8)
        self.decoder = nn.Sequential(
            nn.Unflatten(1, (128, 8, 8)),
            nn.ConvTranspose2d(128, 64, 3),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 3),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 3),
            nn.Sigmoid()
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)
    
    def parameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(mu)
        return mu + eps * std

    def decode(self, z):
        h = self.decoder_input(z)
        return self.decoder(h)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.parameterize(mu, logvar)
        x_recon = self.decode(z)
        return x_recon, mu, logvar
    

def vae_loss(x_recon, x, mu, logvar):
    recon_loss = nn.functional.mse_loss(x_recon, x) #reduction=sum
    kl_loss = mu + logvar
    return recon_loss + kl_loss

In [ ]:
n_epochs = 5
model = VAE()
optimizer = torch.optim.Adam(model.parameters())



for epoch in range(n_epochs):
    model.train()
    total_loss = 0
    for batch, _ in train_dl:
        batch = batch.to(device)
        recon, mu, logvar = model(batch)
        loss = vae_loss(recon, batch, mu, logvar)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() / len(train_dl)
    
    model.eval()
    val_loss = 0

    for batch, _ in val_dl:
        batch = batch.to(device)
        with torch.no_grad():
            recon, mu, logvar = model(batch)
            loss = vae_loss(recon, batch, mu, logvar)
        vae_loss += loss.item() / len(val_dl)
        
    print(f'Epoch {epoch+1}, Train loss = {total_loss:.3f}, Val loss = {val_loss:.3f}')

In [ ]:
import torchvision.utils as vutils
import matplotlib.pyplot as plt

def generate_img(model):
    model.eval()
    with torch.no_grad():
        z = torch.randn(16, 128).to(device)
        samples = model.decode(z).cpu()

    grid = vutils.make_grid(samples, nrow=4)
    plt.imshow(grid.permute(1, 2, 0))
    plt.axis('off')
    plt.show()

'/Users/mhlgvr/Documents/Yandex.Disk.localized/CU2/AI/Bootcamps 2025/memobot'